## **Flowmap City**

https://docs.flowmap.city/docs/create-new-project/prepare-data

On Demand pairs broken down filterable by the hourly temporal space as well as the given routes.

If I map out each currently used stop circa Oct25 and Feb26 I should be able to make a doc with all these used stops. Then from there have the long and lat at each one. The ID can probably be the ID we used in the department, or jsut arbitrarily assign it. Either way, mapping these to the transit lab OD data will be important to be consistent with it, likely can just use a simple dictionary lookup for this.

Data needs to be cleaned up into 2 datasets:

1) locations -> this will be generally static I think I can just use the same one for everything probably

```
id,name,lat,lng/n
1,New York,40.7128,-74.0060
2,Los Angeles,34.0522,-118.2437
3,Chicago,41.8781,-87.6298
4,Houston,29.7604,-95.3698

```

2) flows data -> this will be the expanded origin and destination pairs with count

```
origin,dest,count
1,2,100
1,3,50
1,4,25
2,1,75
2,3,25
2,4,50
```

---

Beginning with Feb26 as a test case here.

In [56]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

data_dir = './data/on-demand/'
df = pd.read_csv(data_dir + 'On Demand Report - User Requests (from 2025-10-01 to 2025-10-31).csv')

print(df.shape)
print(df.info())
df.head(3)


(8507, 59)
<class 'pandas.DataFrame'>
RangeIndex: 8507 entries, 0 to 8506
Data columns (total 59 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Region                          8507 non-null   str    
 1   Day                             8507 non-null   str    
 2   Day of Week                     8507 non-null   str    
 3   User Request Id                 8507 non-null   str    
 4   User Id                         8451 non-null   str    
 5   Rider                           8507 non-null   str    
 6   Rider Email                     44 non-null     str    
 7   Ride Class                      8507 non-null   str    
 8   Driver                          8507 non-null   str    
 9   Driver Id                       8507 non-null   str    
 10  Vehicle                         8507 non-null   str    
 11  Vehicle Id                      8507 non-null   str    
 12  Rider Tags                      0 

,Region,Day,Day of Week,User Request Id,User Id,Rider,Rider Email,Ride Class,Driver,Driver Id,Vehicle,Vehicle Id,Rider Tags,Vehicle Tags,Vehicle Capacity,When Requested,Depart At,Arrive By,Passengers,Wheelchair Passengers,Infant Seat Passengers,Toddler Seat Passengers,Total Passengers,Bicycles,Travel Mode,Start Time,ETA,Pickup Stop,Pickup Stop Tags,Pickup Stop Id,Pickup Location,Dropoff Stop,Dropoff Stop Tags,Dropoff Stop Id,Dropoff Location,Scheduled Pickup,Driver Arrived,Scheduled Dropoff,Actual Pickup,Pickup Passengers,Pickup Wheelchair Passengers,Pickup Infant Seat Passengers,Pickup Toddler Seat Passengers,Pickup Instructions,Pickup Bicycles,Actual Dropoff,State,Isolation,# Pooled Rides,Who Requested,When Canceled,Who Canceled,Why Canceled Reason,Why Canceled,Rider Rating,Rider Unhappy,Rider Feedback,Region ID,Type
0,CABS OnDemand,2025-10-01,Wednesday,080774dd-da0c-4f08-a748-b48c3e00b158,290b7337-e434-4138-b60a-c92ffcd112d3,Hyeryeong Lee,NaN,default,Driver 2,67b09e19-6894-4ea8-bb6f-1541eb98ab4e,OD 2,12e984d1-e039-490a-81b0-55925a69bdb5,NaN,NaN,10,2025-10-01 07:13:00 PM,NaN,NaN,1,0,0,0,1,0,Walking,2025-10-01 07:18:20 PM,2025-10-01 07:25:21 PM,RPAC,NaN,7e1cbfd5-8372-46c0-872e-2045f90fc68f,"40.0005, -83.0180",1165 Kinnear Rd - University Libraries Tech Ce...,No Go,2e6d1e7f-1552-49a7-99b7-3a729743061d,"39.9980, -83.0390",2025-10-01 07:18:20 PM,2025-10-01 07:19:23 PM,2025-10-01 07:25:11 PM,2025-10-01 07:19:30 PM,1.0,0.0,0.0,0.0,NaN,0.0,2025-10-01 07:25:24 PM,Complete,Not Isolated,1.0,Hyeryeong Lee,NaN,NaN,NaN,NaN,5.0,NaN,NaN,ca558ddc-d7f2-4b48-9cac-deea1134f820,On Demand
1,CABS OnDemand,2025-10-01,Wednesday,2697a43d-03f3-46ab-9eb0-3f90c6ce3eeb,1b7349ff-871c-4083-a0c8-7c3f835be3a8,Xu Gao,NaN,default,Driver 2,67b09e19-6894-4ea8-bb6f-1541eb98ab4e,OD 2,12e984d1-e039-490a-81b0-55925a69bdb5,NaN,NaN,10,2025-10-01 07:14:24 PM,NaN,NaN,1,0,0,0,1,0,Walking,2025-10-01 07:31:20 PM,2025-10-01 07:39:49 PM,RPAC,NaN,7e1cbfd5-8372-46c0-872e-2045f90fc68f,"40.0005, -83.0180",Buckeye Village Community Center,NaN,549d9754-c557-43b9-a0d8-9c7715726744,"40.0155, -83.0260",2025-10-01 07:31:20 PM,2025-10-01 07:25:31 PM,2025-10-01 07:38:48 PM,2025-10-01 07:25:33 PM,1.0,0.0,0.0,0.0,NaN,0.0,2025-10-01 07:31:46 PM,Complete,Not Isolated,0.0,Xu Gao,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ca558ddc-d7f2-4b48-9cac-deea1134f820,On Demand
2,CABS OnDemand,2025-10-01,Wednesday,37aa1d01-e999-49e0-9c6d-670ec5dc82b4,5ea97b00-667a-42cd-bc99-c03b846ebe85,Wenyuan Cai,NaN,default,Driver 2,67b09e19-6894-4ea8-bb6f-1541eb98ab4e,OD 2,12e984d1-e039-490a-81b0-55925a69bdb5,NaN,NaN,10,2025-10-01 07:43:31 PM,NaN,NaN,1,0,0,0,1,0,Walking,2025-10-01 07:49:16 PM,2025-10-01 07:57:42 PM,RPAC,NaN,7e1cbfd5-8372-46c0-872e-2045f90fc68f,"40.0005, -83.0180",1165 Kinnear Rd - University Libraries Tech Ce...,No Go,2e6d1e7f-1552-49a7-99b7-3a729743061d,"39.9980, -83.0390",2025-10-01 07:49:16 PM,2025-10-01 07:51:14 PM,2025-10-01 07:55:43 PM,2025-10-01 07:51:17 PM,1.0,0.0,0.0,0.0,NaN,0.0,2025-10-01 07:57:36 PM,Complete,Not Isolated,0.0,Wenyuan Cai,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ca558ddc-d7f2-4b48-9cac-deea1134f820,On Demand


In [57]:
# parse down to necessary cols
df = df[['Day', 'Day of Week', 'Start Time', 'Pickup Stop', 'Pickup Location', 'Dropoff Stop', 'Dropoff Location', 'State']]
print(df.shape)
df.head(3)

(8507, 8)


,Day,Day of Week,Start Time,Pickup Stop,Pickup Location,Dropoff Stop,Dropoff Location,State
0,2025-10-01,Wednesday,2025-10-01 07:18:20 PM,RPAC,"40.0005, -83.0180",1165 Kinnear Rd - University Libraries Tech Ce...,"39.9980, -83.0390",Complete
1,2025-10-01,Wednesday,2025-10-01 07:31:20 PM,RPAC,"40.0005, -83.0180",Buckeye Village Community Center,"40.0155, -83.0260",Complete
2,2025-10-01,Wednesday,2025-10-01 07:49:16 PM,RPAC,"40.0005, -83.0180",1165 Kinnear Rd - University Libraries Tech Ce...,"39.9980, -83.0390",Complete


In [58]:
df.describe().T

,count,unique,top,freq
Day,8507,31,2025-10-29,413
Day of Week,8507,7,Wednesday,1714
Start Time,8507,8379,2025-10-08 06:23:29 PM,3
Pickup Stop,8507,151,Pelotonia Research Center,772
Pickup Location,8507,100,"40.0050, -83.0355",699
Dropoff Stop,8507,128,Pelotonia Research Center,613
Dropoff Location,8507,91,"39.9960, -83.0175",606
State,8507,2,Complete,6539


In [59]:
completed_df = df[df['State'] == 'Complete']

In [60]:
# create a locations dataframe for flowmap city
locations_df = df[['Pickup Stop', 'Pickup Location', 'Dropoff Stop', 'Dropoff Location']]
locations_df = locations_df.drop_duplicates()
print(locations_df['Pickup Stop'].nunique())
print(locations_df['Dropoff Stop'].nunique())

151
128


In [76]:
# Keep each observed stop-name / coordinate combination for auditability
locations_source = pd.concat([
    locations_df[['Pickup Stop', 'Pickup Location']].rename(columns={'Pickup Stop': 'name', 'Pickup Location': 'coord'}),
    locations_df[['Dropoff Stop', 'Dropoff Location']].rename(columns={'Dropoff Stop': 'name', 'Dropoff Location': 'coord'})
]).drop_duplicates().reset_index(drop=True)

print(locations_source.shape)

(158, 2)


In [77]:
# Map alternate labels to a canonical stop name when they refer to the same stop
stop_name_aliases = {
    # 'Alternate stop name': 'Canonical stop name',
}

locations_source[['lat', 'long']] = locations_source['coord'].str.split(',', expand=True)
locations_source['lat'] = locations_source['lat'].astype(float)
locations_source['long'] = locations_source['long'].astype(float)
locations_source['name'] = locations_source['name'].str.strip()
locations_source['canonical_name'] = locations_source['name'].replace(stop_name_aliases)

# Use the median observed coordinate as the representative point for each stop
locations_combined = (
    locations_source.groupby('canonical_name', as_index=False, sort=True)
    .agg(
        name=('canonical_name', 'first'),
        lat=('lat', 'median'),
        long=('long', 'median'),
    )
    [['name', 'lat', 'long']]
    .reset_index(drop=True)
 )

print(locations_combined.shape)

(153, 3)


In [78]:
# Assign one ID per canonical stop
assert locations_combined['name'].is_unique
locations_combined['id'] = range(1, len(locations_combined) + 1)

# Keep IDs first for the Flowmap City locations file
cols = ['id'] + [col for col in locations_combined.columns if col != 'id']
locations_combined = locations_combined[cols]

locations_combined

,id,name,lat,long
0,1,1165 Kinnear Rd - University Libraries Tech Ce...,39.9980,-83.0390
1,2,18th Avenue Library {OVN},40.0010,-83.0130
2,3,1900 Kenny Rd,39.9990,-83.0345
3,4,2000 Kenny Rd,40.0000,-83.0340
4,5,209 W. 18th {OVN},40.0015,-83.0155
...,...,...,...,...
148,149,Wexner Center for the Arts {OVN},40.0005,-83.0085
149,150,Wilce Student Health {OVN},40.0000,-83.0165
150,151,Woody Hayes Athletic Center,40.0130,-83.0230
151,152,Worthington Building {OVN},39.9940,-83.0135


In [85]:
locations_combined[locations_combined.duplicated(subset=['name'], keep=False)]

,id,name,lat,long


In [84]:
# Save canonical locations and the original location variants for auditability
locations_combined.to_csv('on_demand_locations.csv', index=False)
locations_source.to_csv('on_demand_location_variants.csv', index=False)

In [81]:
# Review source coordinate variants that were consolidated by canonical stop name
variant_counts = locations_source.groupby('canonical_name').size()
duplicated_stops = locations_source[
    locations_source['canonical_name'].map(variant_counts) > 1
].sort_values(['canonical_name', 'lat', 'long'])

duplicated_stops[['name', 'canonical_name', 'coord']]

,name,canonical_name,coord
19,Culinary Production Kitchen (CPK),Culinary Production Kitchen (CPK),"40.0145, -83.0350"
37,Culinary Production Kitchen (CPK),Culinary Production Kitchen (CPK),"40.0145, -83.0345"
154,Fisher Commons,Fisher Commons,"40.0050, -83.0340"
39,Fisher Commons,Fisher Commons,"40.0055, -83.0340"
115,Fisher Commons,Fisher Commons,"40.0055, -83.0335"
1,Pelotonia Research Center,Pelotonia Research Center,"40.0050, -83.0355"
43,Pelotonia Research Center,Pelotonia Research Center,"40.0050, -83.0350"
98,Pelotonia Research Center,Pelotonia Research Center,"40.0055, -83.0360"


In [ ]:
locations_source[locations_source['canonical_name'] == 'Pelotonia Research Center'][['name', 'canonical_name', 'coord']]

,id,name,lat,long
1,2,Pelotonia Research Center,40.0050,-83.0355
43,44,Pelotonia Research Center,40.0050,-83.0350
98,99,Pelotonia Research Center,40.0055,-83.0360


In [ ]:
locations_source[locations_source['canonical_name'] == 'Fisher Commons'][['name', 'canonical_name', 'coord']]

,id,name,lat,long
39,40,Fisher Commons,40.0055,-83.0340
115,116,Fisher Commons,40.0055,-83.0335
154,155,Fisher Commons,40.0050,-83.0340


In [ ]:
locations_source[locations_source['canonical_name'] == 'Culinary Production Kitchen (CPK)'][['name', 'canonical_name', 'coord']]

,id,name,lat,long
19,20,Culinary Production Kitchen (CPK),40.0145,-83.0350
37,38,Culinary Production Kitchen (CPK),40.0145,-83.0345



---

### **Flows Data**


In [79]:
# Map each completed trip to exactly one canonical stop ID
stop_name_to_id = locations_combined.set_index('name')['id']

id_completed_df = completed_df.copy()
id_completed_df['origin_id'] = (
    id_completed_df['Pickup Stop']
    .str.strip()
    .replace(stop_name_aliases)
    .map(stop_name_to_id)
 )
id_completed_df['dest_id'] = (
    id_completed_df['Dropoff Stop']
    .str.strip()
    .replace(stop_name_aliases)
    .map(stop_name_to_id)
 )

unmapped_trips = id_completed_df[
    id_completed_df[['origin_id', 'dest_id']].isna().any(axis=1)
 ]
if not unmapped_trips.empty:
    raise ValueError(
        f'{len(unmapped_trips)} completed trips have an unmapped stop name; '
        'add its name to stop_name_aliases or inspect the source data.'
    )

assert len(id_completed_df) == len(completed_df)

id_completed_df

,Day,Day of Week,Start Time,Pickup Stop,Pickup Location,Dropoff Stop,Dropoff Location,State,origin_id,dest_id
0,2025-10-01,Wednesday,2025-10-01 07:18:20 PM,RPAC,"40.0005, -83.0180",1165 Kinnear Rd - University Libraries Tech Ce...,"39.9980, -83.0390",Complete,114,1
1,2025-10-01,Wednesday,2025-10-01 07:31:20 PM,RPAC,"40.0005, -83.0180",Buckeye Village Community Center,"40.0155, -83.0260",Complete,114,21
2,2025-10-01,Wednesday,2025-10-01 07:49:16 PM,RPAC,"40.0005, -83.0180",1165 Kinnear Rd - University Libraries Tech Ce...,"39.9980, -83.0390",Complete,114,1
3,2025-10-01,Wednesday,2025-10-01 08:17:05 PM,Pelotonia Research Center,"40.0050, -83.0355",Center for Electron Microscopy and Analysis,"39.9975, -83.0430",Complete,107,33
5,2025-10-01,Wednesday,2025-10-01 07:20:10 PM,RPAC,"40.0005, -83.0180",1165 Kinnear Rd - University Libraries Tech Ce...,"39.9980, -83.0390",Complete,114,1
...,...,...,...,...,...,...,...,...,...,...
8502,2025-10-31,Friday,2025-10-31 07:45:54 AM,Vehicle Maintenance / Service Annex,"40.0125, -83.0355",Ohio Union {OVN},"39.9980, -83.0095",Complete,143,100
8503,2025-10-31,Friday,2025-10-31 08:46:28 AM,Gateway,"39.9945, -83.0065",Pelotonia Research Center,"40.0050, -83.0350",Complete,55,107
8504,2025-10-31,Friday,2025-10-31 08:26:36 AM,Vehicle Maintenance / Service Annex,"40.0125, -83.0355",Ohio Union {OVN},"39.9980, -83.0095",Complete,143,100
8505,2025-10-31,Friday,2025-10-31 08:51:58 AM,Buckeye Village Community Center,"40.0155, -83.0260",Pelotonia Research Center,"40.0050, -83.0355",Complete,21,107


In [80]:
# Aggregate completed trips and verify no trip counts were lost
origin_dest_counts = (
    id_completed_df.groupby(['origin_id', 'dest_id'])
    .size()
    .reset_index(name='count')
 )

assert origin_dest_counts['count'].sum() == len(completed_df)

origin_dest_counts

,origin_id,dest_id,count
0,1,6,1
1,1,11,2
2,1,14,1
3,1,36,32
4,1,42,29
...,...,...,...
1359,153,128,2
1360,153,129,4
1361,153,132,1
1362,153,136,7


In [67]:
# display top 10 origin-destination pairs by count
origin_dest_counts.sort_values(by='count', ascending=False).head(25)

,origin_id,dest_id,count
733,44,16,214
55,2,16,214
1251,99,16,214
310,16,99,192
294,16,44,192
281,16,2,192
7,1,17,184
315,17,1,146
667,39,3,106
1256,99,36,92


In [83]:
origin_dest_counts.to_csv('on_demand_flows_data.csv', index=False)


---

TODO:

- Go in and clean up the locations, there are multiple entries for the same stop on multiple like Doan Hall/UH Doan, PRC especially here, and CPK
    - Actually maybe a different month would be better here since the new hospital opened in Feb...
- Add temporal aspect of the data, subset by time of day, date, etc
    - I believe this will require SQL queries will look into this